### **Cancellation**

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>در برنامه‌های غیرهمزمان و چندرشته‌ای، گاهی نیاز داریم که یک عملیات در حال اجرا را <strong>متوقف کنیم</strong>، مثلاً در پاسخ به درخواست کاربر. C# مکانیزم استانداردی برای این کار دارد که از <strong>CancellationToken</strong> و <strong>CancellationTokenSource</strong> استفاده می‌کند.</p>

<h3><strong>۱. مکانیزم اولیه‌ی لغو عملیات (<code>CancellationToken</code>)</strong></h3>
<p>یک راه ساده برای پیاده‌سازی لغو عملیات، استفاده از یک <strong>فلگ (flag)</strong> است که مشخص کند آیا عملیات باید متوقف شود یا خیر. مثال زیر یک <strong>کلاس سفارشی <code>CancellationToken</code></strong> را پیاده‌سازی می‌کند:</p>
</div>

In [ ]:
class CancellationToken
{
    public bool IsCancellationRequested { get; private set; }

    public void Cancel() => IsCancellationRequested = true;

    public void ThrowIfCancellationRequested()
    {
        if (IsCancellationRequested)
            throw new OperationCanceledException();
    }
}

//use
async Task Foo(CancellationToken cancellationToken)
{
    for (int i = 0; i < 10; i++)
    {
        Console.WriteLine(i);
        await Task.Delay(1000);  // تأخیر ۱ ثانیه‌ای
        cancellationToken.ThrowIfCancellationRequested();  // بررسی وضعیت لغو
    }
}


<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>🔹 اگر در جایی از برنامه متد <code>Cancel()</code> صدا زده شود، مقدار <code>IsCancellationRequested</code> به <strong><code>true</code></strong> تغییر می‌کند و باعث پرتاب شدن <code>OperationCanceledException</code> در متد <code>Foo</code> می‌شود.</p>
<p>🔹 این پیاده‌سازی دستی، درست کار می‌کند، اما <strong>مشکل همروندی (Thread-Safety) دارد</strong>، زیرا مقدار <code>IsCancellationRequested</code> ممکن است همزمان از چند رشته تغییر کند.</p>

<h3><strong>۲. <code>CancellationTokenSource</code> و <code>CancellationToken</code> در دات‌نت</strong></h3>
<p>دات‌نت مکانیزم استانداردی را با استفاده از <strong>دو کلاس <code>CancellationTokenSource</code> و <code>CancellationToken</code></strong> ارائه می‌دهد:</p>
<ul><li><strong><code>CancellationTokenSource</code></strong>: شیئی است که می‌تواند عملیات لغو را شروع کند.</li><li><strong><code>CancellationToken</code></strong>: شیئی است که به متدهای <code>async</code> ارسال می‌شود تا بررسی کنند که آیا باید متوقف شوند یا خیر.</li></ul>
<h4><strong>نمونه کد استاندارد با <code>CancellationTokenSource</code></strong></h4>
</div>

In [2]:
using System.Threading;

async Task Foo(CancellationToken cancellationToken)
{
    for (int i = 0; i < 10; i++)
    {
        Console.WriteLine(i);
        await Task.Delay(1000);  // تأخیر ۱ ثانیه‌ای
        cancellationToken.ThrowIfCancellationRequested();  // بررسی وضعیت لغو
    }
}
var cancelSource = new CancellationTokenSource();
Task foo = Foo(cancelSource.Token);

await Task.Delay(1000);
// چند ثانیه بعد...
cancelSource.Cancel();  // متوقف کردن عملیات

0
1


<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>🔹 در اینجا، <code>Cancel()</code> باعث می‌شود تمام <code>async</code>هایی که از <code>CancellationToken</code> استفاده می‌کنند، متوقف شوند.</p>
<h3><strong>۳. بهینه‌سازی: لغو فوری <code>Task.Delay</code></strong></h3>
<p>اگر از <code>Task.Delay</code> در کد استفاده کنیم، بدون پاس دادن <code>cancellationToken</code>، ممکن است تا ۱ ثانیه تأخیر بیفتد.<br>برای این‌که عملیات <strong>بلافاصله پس از لغو متوقف شود</strong>، می‌توان <code>cancellationToken</code> را به <code>Task.Delay</code> ارسال کرد:</p>
</div>

In [ ]:
async Task Foo(CancellationToken cancellationToken)
{
    for (int i = 0; i < 10; i++)
    {
        Console.WriteLine(i);
        await Task.Delay(1000, cancellationToken);  // لغو فوری در صورت نیاز
    }
}

In [3]:
using System.Threading;
using System.Threading.Tasks;

var cts = new CancellationTokenSource();

try
{
    Console.WriteLine("Starting Task.Delay for 5 seconds...");
    var task = Task.Delay(5000, cts.Token);

    // منتظر می‌مانیم تا 2 ثانیه بگذرد
    Console.WriteLine("Waiting for 2 seconds...");
    await Task.Delay(2000);

    // لغو توکن پس از 2 ثانیه
    Console.WriteLine("Cancelling the Task.Delay after 2 seconds...");
    cts.Cancel();

    // منتظر می‌مانیم تا Task کامل شود یا لغو شود
    await task;
}
catch (TaskCanceledException)
{
    Console.WriteLine("Task.Delay was cancelled after 2 seconds.");
}
finally
{
    cts.Dispose();
}

Starting Task.Delay for 5 seconds...
Waiting for 2 seconds...
Cancelling the Task.Delay after 2 seconds...
Task.Delay was cancelled after 2 seconds.


In [ ]:
static async Task CustomDelay(int millisecondsDelay, CancellationToken cancellationToken)
{
    var startTime = DateTime.UtcNow;
    var endTime = startTime.AddMilliseconds(millisecondsDelay);

    while (DateTime.UtcNow < endTime)
    {
        // چک کردن CancellationToken
        cancellationToken.ThrowIfCancellationRequested();

        // منتظر می‌مانیم تا زمان کوتاهی بگذرد (مثلاً 100 میلی‌ثانیه)
        await Task.Delay(100); // این تاخیر کوچک برای جلوگیری از مصرف زیاد CPU است
    }
}

#### **`Cancellation`** in **ASP.NET Core**

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>در <strong>ASP.NET Core</strong>، زمانی که کلاینت یک درخواست را به سرور ارسال می‌کند، ممکن است نیاز باشد که <strong>درخواست را لغو کنیم</strong> (مثلاً وقتی که کاربر صفحه را ببندد یا درخواست زمان زیادی ببرد). این قابلیت با استفاده از <strong><code>CancellationToken</code> و <code>CancellationTokenSource</code></strong> مدیریت می‌شود.</p>
<h3><strong>۱. ایجاد <code>CancellationTokenSource</code> توسط Kestrel</strong></h3>
<p>وقتی که Kestrel (وب‌سرور پیش‌فرض در ASP.NET Core) یک درخواست HTTP را دریافت می‌کند، به‌صورت <strong>خودکار</strong> یک <code>CancellationTokenSource</code> برای آن درخواست ایجاد کرده و آن را داخل <code>HttpContext.RequestAborted</code> قرار می‌دهد.</p>
<p>✅ اگر کلاینت <strong>اتصال را ببندد</strong> یا <strong>مدت‌زمان درخواست بیش از حد شود</strong>، Kestrel مقدار <code>IsCancellationRequested</code> را در <code>CancellationTokenSource</code> روی <code>true</code> تنظیم کرده و <strong>تمام پردازش‌های مرتبط را متوقف می‌کند</strong>.</p>
<p>✅ این <code>CancellationTokenSource</code> توسط ASP.NET Core <strong>مدیریت شده و نیازی به ساخت دستی آن نداریم</strong>.</p>
<h3><strong>۲. دریافت <code>CancellationToken</code> در کنترلر</strong></h3>
<p>در کنترلرهای ASP.NET Core، دو روش برای دریافت <code>CancellationToken</code> وجود دارد:</p>
<h4>✅ <strong>روش پیشنهادی (استفاده از پارامتر <code>CancellationToken</code>)</strong></h4>
<p><code>ASP.NET Core</code> به‌صورت خودکار <code>HttpContext.RequestAborted</code> را به این متغیر مقداردهی می‌کند:</p>
</div>

In [ ]:
[HttpGet("{id}")]
public async Task<IActionResult> GetData(int id, CancellationToken cancellationToken)
{
    return await _mediator.Send(new GetDataQuery { Id = id }, cancellationToken);
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>✅ این روش <strong>ساده و خوانا</strong> است و <code>ASP.NET Core</code> مقدار <code>RequestAborted</code> را به <code>cancellationToken</code> پاس می‌دهد.</p>
<h4>🔹 <strong>روش جایگزین (استفاده مستقیم از <code>HttpContext.RequestAborted</code>)</strong></h4>
<p>می‌توان <code>CancellationToken</code> را مستقیماً از <code>HttpContext</code> دریافت کرد:</p>
</div>

In [ ]:
[HttpGet("{id}")]
public async Task<IActionResult> GetData(int id)
{
    var cancellationToken = HttpContext.RequestAborted;
    return await _mediator.Send(new GetDataQuery { Id = id }, cancellationToken);
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>✅ <strong>مزیت:</strong> کنترل بیشتری روی <code>HttpContext</code> داریم.<br>❌ <strong>عیب:</strong> کد خوانایی کمتری دارد و استفاده از پارامتر <code>CancellationToken</code> پیشنهاد می‌شود.</p>
<h3><strong>۳. دریافت <code>CancellationToken</code> در <code>MediatR</code></strong></h3>
<p>در <strong>CQRS و MediatR</strong>، <code>CancellationToken</code> به‌طور خودکار به <code>Handle</code> ارسال می‌شود. کافی است آن را در <strong>هندلر</strong> دریافت کرده و به سرویس‌ها پاس دهیم:</p>
</div>

In [ ]:
public class GetDataQuery : IRequest<string>
{
    public int Id { get; set; }
}

public class GetDataQueryHandler : IRequestHandler<GetDataQuery, string>
{
    private readonly IDataService _dataService;

    public GetDataQueryHandler(IDataService dataService)
    {
        _dataService = dataService;
    }

    public async Task<string> Handle(GetDataQuery request, CancellationToken cancellationToken)
    {
        cancellationToken.ThrowIfCancellationRequested();
        
        return await _dataService.GetDataAsync(request.Id, cancellationToken);
    }
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>✅ <code>CancellationToken</code> به سرویس‌ها پاس داده شده تا اگر کاربر <strong>درخواست را لغو کرد، پردازش بیهوده انجام نشود.</strong></p>
<h4><strong>۴. استفاده از <code>CancellationToken</code> در سرویس‌ها و دیتابیس</strong></h4>
<p>اگر سرویس یا <strong>دیتابیس کوئری‌های سنگین</strong> اجرا می‌کند، بهتر است <code>CancellationToken</code> را در متدهای async پاس دهیم:</p>
</div>

In [ ]:
public class DataService : IDataService
{
    private readonly ApplicationDbContext _context;

    public DataService(ApplicationDbContext context)
    {
        _context = context;
    }

    public async Task<string> GetDataAsync(int id, CancellationToken cancellationToken)
    {
        return await _context.Data
                             .Where(d => d.Id == id)
                             .Select(d => d.Name)
                             .FirstOrDefaultAsync(cancellationToken); // اعمال توکن در EF Core
    }
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>✅ <code>Entity Framework Core</code> از <code>CancellationToken</code> پشتیبانی می‌کند، بنابراین اگر کاربر درخواست را لغو کند، <strong>کوئری متوقف می‌شود</strong>.</p>
<h3><strong>۵. استفاده از <code>CancellationToken</code> در <code>Task.Delay</code> و <code>HttpClient</code></strong></h3>
<p>اگر در درخواست‌های خارجی یا <strong>تأخیر‌های برنامه (Delay)</strong> استفاده می‌کنید، می‌توان <code>CancellationToken</code> را به <code>Task.Delay</code> یا <code>HttpClient</code> پاس داد:</p>
</div>

In [ ]:
public async Task<string> CallExternalApi(CancellationToken cancellationToken)
{
    using var client = new HttpClient();
    var response = await client.GetAsync("https://api.example.com", cancellationToken);
    return await response.Content.ReadAsStringAsync();
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>✅ اگر درخواست لغو شود، <code>HttpClient</code> دیگر منتظر دریافت پاسخ نمی‌ماند و متوقف می‌شود.</p>
<h3><strong>۶. مدیریت <code>OperationCanceledException</code> برای جلوگیری از خطاهای غیرضروری</strong></h3>
<p>اگر درخواست لغو شود، ممکن است <strong>Exception</strong> پرتاب شود. می‌توان آن را مدیریت کرد:</p>
</div>

In [ ]:
public async Task<IActionResult> GetData(int id, CancellationToken cancellationToken)
{
    try
    {
        return await _mediator.Send(new GetDataQuery { Id = id }, cancellationToken);
    }
    catch (OperationCanceledException)
    {
        return StatusCode(499, "درخواست توسط کلاینت لغو شد"); // 499 = Client Closed Request
    }
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>✅ کد <strong>HTTP 499</strong> نشان می‌دهد که درخواست <strong>توسط کلاینت لغو شده</strong> است.</p>
<h3><strong>۷. چه زمانی <code>CancellationToken</code> کار نمی‌کند؟</strong></h3>
<p>۱. <strong>اگر از <code>CancellationToken</code> در عملیات‌ها استفاده نکنید، پردازش حتی بعد از لغو درخواست ادامه می‌یابد!</strong><br>۲. <strong>اگر متدهای sync استفاده شوند (<code>.Result</code> یا <code>.Wait()</code> در <code>Task</code>)، <code>CancellationToken</code> تأثیری نخواهد داشت.</strong><br>۳. <strong>اگر <code>CancellationToken</code> را در دیتابیس یا پردازش‌های async پاس ندهید، عملیات همچنان ادامه خواهد یافت.</strong></p>
</div>

### **Progress Reporting**

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>زمانی که یک عملیات <strong>زمان‌بر و غیربلاک‌کننده</strong> اجرا می‌شود، ممکن است نیاز داشته باشیم که <strong>میزان پیشرفت</strong> آن را به کاربر نمایش دهیم. در <code>C#</code> دو روش اصلی برای این کار وجود دارد:</p>
<ol><li><strong>استفاده از <code>Action&lt;T&gt;</code></strong> (مناسب برای <strong>برنامه‌های کنسولی</strong>)</li><li><strong>استفاده از <code>IProgress&lt;T&gt;</code> و <code>Progress&lt;T&gt;</code></strong> (مناسب برای <strong>برنامه‌های UI و ASP.NET Core</strong>)</li></ol>

<h3><strong>۱. روش ساده با <code>Action&lt;T&gt;</code> (برای کنسول)</strong></h3>
<p>در این روش، یک <strong>delegate از نوع <code>Action&lt;int&gt;</code></strong> را به متد <code>Foo</code> پاس می‌دهیم. هر بار که مقدار پیشرفت تغییر می‌کند، این delegate صدا زده می‌شود:</p>
</div>

In [2]:
Task Foo(Action<int> onProgressPercentChanged)
{
    return Task.Run(() =>
    {
        for (int i = 0; i < 1000; i++)
        {
            if (i % 10 == 0) onProgressPercentChanged(i / 10);
            // انجام پردازش سنگین...
        }
    });
}

// ✅ در برنامه کنسولی می‌توان این متد را به این شکل فراخوانی کرد:
Action<int> progress = i => Console.WriteLine(i + " %");
await Foo(progress);


0 %
1 %
2 %
3 %
4 %
5 %
6 %
7 %
8 %
9 %
10 %
11 %
12 %
13 %
14 %
15 %
16 %
17 %
18 %
19 %
20 %
21 %
22 %
23 %
24 %
25 %
26 %
27 %
28 %
29 %
30 %
31 %
32 %
33 %
34 %
35 %
36 %
37 %
38 %
39 %
40 %
41 %
42 %
43 %
44 %
45 %
46 %
47 %
48 %
49 %
50 %
51 %
52 %
53 %
54 %
55 %
56 %
57 %
58 %
59 %
60 %
61 %
62 %
63 %
64 %
65 %
66 %
67 %
68 %
69 %
70 %
71 %
72 %
73 %
74 %
75 %
76 %
77 %
78 %
79 %
80 %
81 %
82 %
83 %
84 %
85 %
86 %
87 %
88 %
89 %
90 %
91 %
92 %
93 %
94 %
95 %
96 %
97 %
98 %
99 %


<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>🔹 این روش <strong>در برنامه‌های کنسولی کار می‌کند</strong> اما در <strong>برنامه‌های UI (مانند WPF، WinForms، Blazor) یا ASP.NET Core</strong> باعث مشکلات <strong>Thread-Safety</strong> می‌شود، زیرا اجرای این delegate از یک <strong>Thread Worker</strong> انجام می‌شود و ممکن است به <strong>UI Thread</strong> دسترسی مستقیم نداشته باشد.</p>
</div>

#### **`IProgress<T> and Progress<T>`**

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>در این روش، به‌جای <code>Action&lt;T&gt;</code> از <code>IProgress&lt;T&gt;</code> استفاده می‌کنیم.<br>کلاس <code>Progress&lt;T&gt;</code> مقدار <strong><code>SynchronizationContext</code> (Context فعلی UI یا درخواست HTTP)</strong> را <strong>ذخیره</strong> می‌کند تا <code>Report()</code> همیشه در <strong>Thread مناسب</strong> اجرا شود.</p>
<h4><strong>🔹 بازنویسی متد <code>Foo</code> با <code>IProgress&lt;T&gt;</code></strong></h4>
</div>

In [ ]:
Task Foo(IProgress<int> progress)
{
    return Task.Run(() =>
    {
        for (int i = 0; i < 1000; i++)
        {
            if (i % 10 == 0) progress.Report(i / 10);
            // انجام پردازش سنگین...
        }
    });
}

var progress = new Progress<int>(i => Console.WriteLine(i + " %"));
await Foo(progress);


<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h3><strong> نحوه استفاده از <code>IProgress&lt;T&gt;</code> در ASP.NET Core</strong></h3>
<p>در <strong>ASP.NET Core</strong>، چون درخواست HTTP معمولاً <strong>Context مشخصی ندارد</strong>، می‌توان مقدار پیشرفت را از طریق <strong>SignalR یا Logging</strong> ارسال کرد.</p>
</div>

In [ ]:
[HttpGet("process")]
public async Task<IActionResult> StartProcess(CancellationToken cancellationToken)
{
    var progress = new Progress<int>(percent =>
    {
        _logger.LogInformation($"Progress: {percent} %");
    });

    await Foo(progress);

    return Ok("Processing started...");
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h3><strong> ارسال مقدار پیشرفت به کلاینت با <code>SignalR</code></strong></h3>
<p>در <strong>ASP.NET Core + Blazor + JavaScript</strong>، می‌توان مقدار پیشرفت را <strong>از طریق WebSockets و SignalR</strong> برای کلاینت ارسال کرد:</p>
<h4><strong>🔹 مثال: ارسال مقدار پیشرفت به کلاینت با SignalR</strong></h4>
</div>

In [ ]:
public class ProgressHub : Hub { }

public class LongRunningService
{
    private readonly IHubContext<ProgressHub> _hubContext;

    public LongRunningService(IHubContext<ProgressHub> hubContext)
    {
        _hubContext = hubContext;
    }

    public async Task RunProcess(CancellationToken cancellationToken)
    {
        var progress = new Progress<int>(async percent =>
        {
            await _hubContext.Clients.All.SendAsync("ReceiveProgress", percent);
        });

        await Foo(progress);
    }
}

### **The Task-Based Asynchronous Pattern**

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h3>۱. <strong>الگوی ناهمزمانی مبتنی بر تسک (TAP) چیست؟</strong></h3>
<p>TAP یک الگوی طراحی است که برای پیاده‌سازی عملیات‌های ناهمزمان در .NET استفاده می‌شود. این الگو از کلاس <code>Task</code> و <code>Task&lt;TResult&gt;</code> برای نشان دادن عملیات‌های ناهمزمان استفاده می‌کند. وقتی یک متد ناهمزمان را فراخوانی می‌کنید، این متد یک <code>Task</code> برمی‌گرداند که نشان‌دهنده‌ی یک عملیات در حال اجرا است.</p>
<h3>۲. <strong>ویژگی‌های اصلی متدهای TAP</strong></h3>
<p>متدهایی که از الگوی TAP پیروی می‌کنند، معمولاً ویژگی‌های زیر را دارند:</p>
<h4><strong>الف) بازگشت یک تسک "داغ" (Hot Task)</strong></h4>
<ul><li><p>متدهای TAP یک <code>Task</code> یا <code>Task&lt;TResult&gt;</code> برمی‌گردانند که به آن <strong>"داغ" (Hot)</strong> گفته می‌شود. یعنی تسک بلافاصله پس از فراخوانی متد شروع به اجرا می‌کند و نیازی به فراخوانی متد <code>Start</code> نیست.</p></li></ul>
<h4><strong>ب) پسوند "Async"</strong></h4>
<ul><li><p>نام این متدها معمولاً با پسوند <strong>"Async"</strong> پایان می‌یابد (مثلاً <code>ReadAsync</code> یا <code>WriteAsync</code>). این پسوند نشان‌دهنده‌ی این است که متد یک عملیات ناهمزمان را انجام می‌دهد. البته استثناهایی هم وجود دارد، مثلاً در مورد متدهایی که ترکیب‌کننده‌ی تسک‌ها (Task Combinators) هستند، ممکن است این پسوند استفاده نشود.</p></li></ul>
<h4><strong>ج) پشتیبانی از لغو (Cancellation) و گزارش پیشرفت (Progress Reporting)</strong></h4>
<ul><li><p>متدهای TAP معمولاً از <strong>توکن لغو (CancellationToken)</strong> و/یا <strong>گزارش پیشرفت (IProgress<span class="ds-markdown-html">&lt;T&gt;</span>)</strong> پشتیبانی می‌کنند. این ویژگی‌ها به شما امکان می‌دهند عملیات ناهمزمان را در صورت نیاز لغو کنید یا پیشرفت آن را دنبال کنید.</p></li></ul>
<h4><strong>د) بازگشت سریع به caller</strong></h4>
<ul><li><p>متدهای TAP باید به سرعت به caller بازگردند. یعنی بخش همزمان (Synchronous) اولیه‌ی متد باید کوتاه باشد تا بلوک‌کننده نباشد. این ویژگی باعث می‌شود که متد به‌طور موثر از منابع سیستم استفاده کند.</p></li></ul>
<h4><strong>ه) عدم استفاده از thread در عملیات I/O-bound</strong></h4>
<ul><li><p>اگر عملیات ناهمزمان از نوع <strong>I/O-bound</strong> باشد (مثلاً خواندن از فایل یا شبکه)، متد TAP نیازی به اشغال یک thread جداگانه ندارد. این کار باعث بهبود عملکرد و کاهش مصرف منابع می‌شود.</p></li></ul>
<h3>۳. <strong>چرا TAP مهم است؟</strong></h3>
<p>TAP یک الگوی استاندارد و ساده برای پیاده‌سازی عملیات‌های ناهمزمان است. با استفاده از این الگو، می‌توانید کدهای ناهمزمان بنویسید که:</p>
<ul><li><p>خوانایی بالایی دارد.</p></li><li><p>به‌راحتی قابل نگهداری است.</p></li><li><p>از منابع سیستم به‌طور بهینه استفاده می‌کند.</p></li></ul>

<p>الگوی TAP یک روش استاندارد و کارآمد برای پیاده‌سازی عملیات‌های ناهمزمان در .NET است. با پیروی از این الگو، می‌توانید کدهایی بنویسید که همزمانی (Concurrency) را به‌طور موثر مدیریت می‌کند و از منابع سیستم بهینه‌تر استفاده می‌کند. استفاده از <code>async</code> و <code>await</code> در C# نیز نوشتن چنین کدهایی را بسیار ساده می‌کند.</p>
</div>

In [ ]:
public async Task<int> ReadFileAsync(string path)
{
    using (var reader = new StreamReader(path))
    {
        string content = await reader.ReadToEndAsync();
        return content.Length;
    }
}

### **Task Combinators**

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<h3>۱. <strong>ترکیب‌کننده‌های تسک (Task Combinators) چیست؟</strong></h3>
<p>ترکیب‌کننده‌های تسک، توابعی هستند که تسک‌ها را به‌صورت مفید ترکیب می‌کنند، بدون اینکه به ماهیت یا کاری که آن تسک‌ها انجام می‌دهند وابسته باشند. این ترکیب‌کننده‌ها به شما کمک می‌کنند تا چندین تسک را به‌صورت موثر مدیریت کنید، مثلاً منتظر بمانید تا اولین تسک کامل شود یا همه‌ی تسک‌ها به پایان برسند.</p>

<h3>۲. <strong>ترکیب‌کننده‌های اصلی: <code>Task.WhenAny</code> و <code>Task.WhenAll</code></strong></h3>
<p>در .NET، دو ترکیب‌کننده‌ی اصلی وجود دارد:</p>
<h4><strong>الف) <code>Task.WhenAny</code></strong></h4>
<ul><li><p>این تابع یک تسک برمی‌گرداند که وقتی <strong>هر یک از تسک‌های ورودی</strong> کامل شود، کامل می‌شود.</p></li><li><p>به عبارت دیگر، <code>Task.WhenAny</code> منتظر می‌ماند تا اولین تسک از مجموعه‌ی تسک‌ها به پایان برسد.</p></li><li><p>این تابع برای سناریوهایی مفید است که می‌خواهید به محض اتمام اولین تسک، ادامه‌ی کار را اجرا کنید.</p></li></ul>

</div>

In [ ]:
async Task<int> Delay1() { await Task.Delay (1000); return 1; }
async Task<int> Delay2() { await Task.Delay (2000); return 2; }
async Task<int> Delay3() { await Task.Delay (3000); return 3; }

Task<int> winningTask = await Task.WhenAny(Delay1(), Delay2(), Delay3());
Console.WriteLine("Done");
Console.WriteLine(winningTask.Result); // 1

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>در این مثال:</p>
<ul><li><p><code>Task.WhenAny</code> منتظر می‌ماند تا اولین تسک از <code>Delay1</code>, <code>Delay2</code>, و <code>Delay3</code> کامل شود.</p></li><li><p>چون <code>Delay1</code> زودتر از بقیه کامل می‌شود (بعد از ۱ ثانیه)، <code>winningTask</code> به <code>Delay1</code> اشاره می‌کند.</p></li><li><p>نتیجه‌ی <code>winningTask</code> (یعنی <code>1</code>) چاپ می‌شود.</p></li></ul>
<p><strong>نکته‌ها:</strong></p>
<ul><li><p>بهتر است به جای دسترسی مستقیم به <code>Result</code>، از <code>await</code> استفاده کنید تا اگر تسک با خطا مواجه شد، خطا به‌درستی منتشر شود.</p></li><li><p>اگر تسک‌های دیگر بعداً با خطا مواجه شوند، خطاهای آن‌ها نادیده گرفته می‌شود، مگر اینکه صریحاً آن‌ها را await کنید یا خاصیت <code>Exception</code> آن‌ها را بررسی کنید.</p></li></ul>

<p><strong>استفاده برای Timeout یا لغو:</strong></p>
</div>

In [ ]:
Task<string> task = SomeAsyncFunc();
Task winner = await Task.WhenAny(task, Task.Delay(5000));

if (winner != task) throw new TimeoutException();
string result = await task; // Unwrap result/re-throw

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>در این مثال:</p>
<ul><li><p>اگر <code>SomeAsyncFunc</code> قبل از ۵ ثانیه کامل شود، نتیجه‌ی آن برگردانده می‌شود.</p></li><li><p>در غیر این صورت، یک استثنای <code>TimeoutException</code> پرتاب می‌شود.</p></li></ul>
<h4><strong>ب) <code>Task.WhenAll</code></strong></h4>
<ul><li><p>این تابع یک تسک برمی‌گرداند که وقتی <strong>همه‌ی تسک‌های ورودی</strong> کامل شوند، کامل می‌شود.</p></li><li><p>به عبارت دیگر، <code>Task.WhenAll</code> منتظر می‌ماند تا تمام تسک‌ها به پایان برسند.</p></li><li><p>این تابع برای سناریوهایی مفید است که می‌خواهید چندین عملیات ناهمزمان را به‌صورت موازی اجرا کنید و منتظر بمانید تا همه‌ی آن‌ها کامل شوند.</p></li></ul>
<p><strong>مثال:</strong></p>
</div>

In [ ]:
await Task.WhenAll(Delay1(), Delay2(), Delay3());

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>در این مثال:</p>
<ul><li><p><code>Task.WhenAll</code> منتظر می‌ماند تا هر سه تسک <code>Delay1</code>, <code>Delay2</code>, و <code>Delay3</code> کامل شوند.</p></li><li><p>چون طولانی‌ترین تسک (<code>Delay3</code>) ۳ ثانیه طول می‌کشد، کل فرآیند ۳ ثانیه زمان می‌برد.</p></li></ul>
<p><strong>تفاوت با await کردن تسک‌ها به‌صورت جداگانه:</strong><br>اگر تسک‌ها را به‌صورت جداگانه await کنید:</p>
</div>

In [ ]:
Task task1 = Delay1(), task2 = Delay2(), task3 = Delay3();
await task1; await task2; await task3;

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<ul><li><p>اگر <code>task1</code> با خطا مواجه شود، <code>task2</code> و <code>task3</code> اصلاً await نمی‌شوند و خطاهای آن‌ها نادیده گرفته می‌شود.</p></li><li><p>اما <code>Task.WhenAll</code> منتظر می‌ماند تا همه‌ی تسک‌ها کامل شوند، حتی اگر برخی از آن‌ها با خطا مواجه شوند.</p></li></ul>
<p><strong>مدیریت خطاها در <code>Task.WhenAll</code>:</strong><br>اگر چندین تسک با خطا مواجه شوند، خطاها در یک <code>AggregateException</code> ترکیب می‌شوند. برای دسترسی به همه‌ی خطاها:</p>
</div>

In [ ]:
Task task1 = Task.Run(() => { throw null; });
Task task2 = Task.Run(() => { throw null; });
Task all = Task.WhenAll(task1, task2);
try { await all; }
catch
{
    Console.WriteLine(all.Exception.InnerExceptions.Count); // 2
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p><strong>استفاده با تسک‌هایی که نتیجه برمی‌گردانند (<code>Task&lt;TResult&gt;</code>):</strong><br>اگر تسک‌ها نتیجه برگردانند، <code>Task.WhenAll</code> یک آرایه از نتایج (<code>TResult[]</code>) برمی‌گرداند:</p>
</div>

In [ ]:
Task<int> task1 = Task.Run(() => 1);
Task<int> task2 = Task.Run(() => 2);
int[] results = await Task.WhenAll(task1, task2); // { 1, 2 }

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p><strong>مثال عملی: دانلود موازی چند URI و محاسبه‌ی مجموع طول داده‌ها:</strong></p>
</div>

In [ ]:
async Task<int> GetTotalSize(string[] uris)
{
    IEnumerable<Task<byte[]>> downloadTasks = uris.Select(uri =>
        new WebClient().DownloadDataTaskAsync(uri));
    byte[][] contents = await Task.WhenAll(downloadTasks);
    return contents.Sum(c => c.Length);
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn; text-align:justify">
<p>در این مثال:</p>
<ul><li><p>داده‌های چندین URI به‌صورت موازی دانلود می‌شوند.</p></li><li><p>پس از اتمام همه‌ی دانلودها، مجموع طول داده‌ها محاسبه و برگردانده می‌شود.</p></li></ul>
<p><strong>بهینه‌سازی:</strong><br>برای جلوگیری از نگه‌داشت‌ن غیرضروری داده‌ها در حافظه، می‌توانید طول داده‌ها را بلافاصله پس از دانلود محاسبه کنید:</p>

</div>

In [ ]:
async Task<int> GetTotalSize(string[] uris)
{
    IEnumerable<Task<int>> downloadTasks = uris.Select(async uri =>
        (await new WebClient().DownloadDataTaskAsync(uri)).Length);
    int[] contentLengths = await Task.WhenAll(downloadTasks);
    return contentLengths.Sum();
}